# California Historic Vineyards – Data Extraction & Analysis

**Source:** [Historic Vineyard Society](https://historicvineyardsociety.org/vineyards)

This notebook:
1. Scrapes all vineyard pages (or loads cached/sample data)
2. Extracts structured table data (AVA, County, Owner, …)
3. Extracts free-text (Characteristics, Description)
4. Analyses text to identify **grape varieties** (with percentages) and **soil types**
5. Visualises variety composition as pie charts and stacked bar charts

## 1. Setup

In [ ]:
import json, os, sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Make sure local modules are importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from analyzer   import analyse_all, extract_grape_varieties, extract_soil_types, apply_ttb_rule
from visualizer import (
    plot_vineyard_pie,
    plot_all_pies,
    plot_stacked_bar,
    plot_interactive_bar,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## 2. Data Acquisition

Set `USE_SAMPLE = True` to use the bundled sample data (10 vineyards),  
or `False` to run the live scraper (requires internet access).

In [ ]:
USE_SAMPLE = True   # ← flip to False to run the live scraper

if USE_SAMPLE:
    with open('sample_data.json', 'r', encoding='utf-8') as fh:
        records = json.load(fh)
    print(f'Loaded {len(records)} sample vineyard records.')
else:
    from scraper import VineyardScraper
    scraper = VineyardScraper(delay=2.0)
    records = scraper.scrape_all()
    os.makedirs('data', exist_ok=True)
    with open('data/vineyards_raw.json', 'w', encoding='utf-8') as fh:
        json.dump(records, fh, indent=2, ensure_ascii=False)
    print(f'Scraped {len(records)} vineyards → data/vineyards_raw.json')

## 3. Structured Data Preview

In [ ]:
# Convert records to a quick preview DataFrame
preview_fields = ['name', 'AVA', 'County', 'Decade', 'Current Owner', 'Planted by', 'Wineries']
preview_df = pd.DataFrame([
    {f: r.get(f, '') for f in preview_fields}
    for r in records
])
preview_df

## 4. Text Analysis – Grape Varieties & Soil Types

In [ ]:
# Run the full analysis
df = analyse_all(records)
df.head(3)

In [ ]:
# Show variety / soil summary
summary = df[['name', 'ava', 'county', 'decade', 'ttb_variety', 'soil_types']].copy()
summary

### 4a. Example: Alegría Vineyard

The text states: *"78% Zinfandel, 11% Alicante Bouschet, 9% Petite Sirah.  
The remaining 2% includes Carignan, Trousseau (Bastardo), Sangiovese, …"*

In [ ]:
# Inspect one vineyard in detail
target_name = 'Alegría Vineyard'
row = next((r for r in records if r['name'] == target_name), None)

if row:
    text = " ".join(str(row.get(k, '')) for k in ['description', 'Characteristics'])
    varieties = extract_grape_varieties(text)
    soils     = extract_soil_types(text)
    ttb       = apply_ttb_rule(varieties)

    print(f"Vineyard : {target_name}")
    print(f"TTB label: {ttb or 'none (field blend)'} (≥75 % rule)")
    print(f"Soils    : {', '.join(soils) or 'not detected'}")
    print("\nVarieties:")
    for var, pct in sorted(varieties.items(), key=lambda x: (x[1] or 0), reverse=True):
        print(f"  {var:30s} {str(round(pct, 2))+'%' if pct is not None else 'unknown %'}")

## 5. Visualisations

### 5a. Pie chart – single vineyard

In [ ]:
row_data = df[df['name'] == 'Alegría Vineyard'].iloc[0]
fig = plot_vineyard_pie(
    name=row_data['name'],
    varieties_json=row_data['varieties_json'],
    ttb_variety=row_data['ttb_variety'],
)
plt.show()

### 5b. Grid of pie charts – all vineyards

In [ ]:
fig = plot_all_pies(df, cols=3)
plt.show()

### 5c. Stacked horizontal bar chart – all vineyards

In [ ]:
fig = plot_stacked_bar(df)
plt.show()

### 5d. Interactive Plotly chart (rendered inline)

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

plot_interactive_bar(df)  # shows inline in notebook

## 6. Soil Type Distribution

In [ ]:
# Tally soil types across all vineyards
from collections import Counter

soil_counter = Counter()
for soils_str in df['soil_types'].dropna():
    for soil in soils_str.split('; '):
        if soil.strip():
            soil_counter[soil.strip()] += 1

soil_df = pd.DataFrame(soil_counter.most_common(), columns=['Soil Type', 'Count'])
display(soil_df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(soil_df['Soil Type'], soil_df['Count'], color='#8B4513')
ax.set_xlabel('Number of Vineyards')
ax.set_title('Soil Types Across Historic Vineyards')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Most Common Grape Varieties

In [ ]:
variety_presence = Counter()
for vjson in df['varieties_json']:
    try:
        varieties = json.loads(vjson)
        for var, pct in varieties.items():
            if pct is not None and pct > 0:
                variety_presence[var] += 1
    except Exception:
        pass

top_varieties = pd.DataFrame(
    variety_presence.most_common(15),
    columns=['Variety', 'Vineyards']
)
display(top_varieties)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_varieties['Variety'], top_varieties['Vineyards'], color='#722F37')
ax.set_xlabel('Number of Vineyards')
ax.set_title('Most Common Grape Varieties Across Historic Vineyards')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Save All Outputs

In [ ]:
os.makedirs('output', exist_ok=True)

# Save CSV
df.to_csv('output/vineyards_analysed.csv', index=False)

# Save charts
plot_all_pies(df, output_path='output/pies_grid.png', cols=3)
plot_stacked_bar(df, output_path='output/stacked_bar.png')
plot_interactive_bar(df, output_path='output/interactive_bar.html')

print('Saved to output/')